##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Kakunin + Google Agent Development Kit (ADK) Compliance Playground

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/google-adk/kakunin_agent_compliance/google_adk_playground.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook demonstrates how to build and execute compliance-guarded AI agents with Google Agent Development Kit (ADK) and Kakunin.

### Step 1: Install Dependencies
First, install `kakunin`, `google-adk`, and `python-dotenv`.

In [ ]:
%pip install -q kakunin google-adk python-dotenv


### Step 2: Configure Environment Keys
Set your Kakunin and Gemini API keys below.

In [ ]:
import os
import getpass

os.environ["KAK_API_KEY"] = getpass.getpass("Enter Kakunin API Key (kak_live_...): ")
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key (AIzaSy...): ")

### Step 3: Run the Agent Integration Demo
Now, run a script to register the agent with Kakunin, issue its compliance certificate, bind tool scopes, and run the agent.

In [ ]:
# @title Model Selection
GEMINI_MODEL_ID = "gemini-3.5-flash" # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

import asyncio
from google.adk import Agent, LocalAgentConfig
from kakunin import Kakunin
from kakunin.exceptions import ScopeViolationError
from kakunin.integrations.google_antigravity import get_kakunin_hooks

# Define mock financial tools
def query_market_prices(symbol: str) -> str:
    """Query current prices for a given ticker symbol."""
    print(f"[Tool Executing] query_market_prices: {symbol}")
    return f"Latest price for {symbol}: $150.00"

def execute_market_trade(symbol: str, amount: int) -> str:
    """Execute a market buy order for a symbol."""
    print(f"[Tool Executing] execute_market_trade: Buying {amount} of {symbol}")
    return f"Successfully executed trade: Buy {amount} shares of {symbol}"

async def run_compliance_demo():
    async with Kakunin(api_key=os.environ["KAK_API_KEY"]) as kakunin_client:
        # 1. Register agent
        agent_record = await kakunin_client.agents.create(
            name="NotebookTrader",
            model=GEMINI_MODEL_ID,
            version="1.0.0",
            model_hash="sha256:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
            metadata={"permitted_actions": ["market.read", "trade.execute"]}
        )
        print(f"Registered Agent: {agent_record.id}")

        # 2. Issue certificate
        cert = await kakunin_client.agents.certify(agent_record.id)
        print(f"Issued Certificate Serial: {cert.serial_number}")

        # 3. Create compliance hooks with scope mapping
        hooks = get_kakunin_hooks(
            kakunin=kakunin_client,
            agent_id=agent_record.id,
            tool_scopes_mapping={
                "query_market_prices": ["market.read"],
                "execute_market_trade": ["trade.execute"]
            }
        )

        # 4. Initialize ADK agent
        config = LocalAgentConfig(
            model=GEMINI_MODEL_ID,
            system_instructions="You are a helpful assistant with financial tools.",
            tools=[query_market_prices, execute_market_trade],
            hooks=hooks
        )

        async with Agent(config=config) as agent:
            print("\n--- Query Price ---")
            try:
                res = await agent.chat("Check price of GOOG")
                print(f"Response: {res}")
            except ScopeViolationError as e:
                print(f"Compliance block: {e}")

            print("\n--- Execute Trade ---")
            try:
                res = await agent.chat("Buy 5 shares of GOOG")
                print(f"Response: {res}")
            except ScopeViolationError as e:
                print(f"Compliance block: {e}")

await run_compliance_demo()